# SageMaker Studio Demo: Pipeline

A training pipeline that runs on SageMaker compute as a DAG.

```
preprocess -> train -> evaluate -> [rmse gate] -> register
```

1. upsert the pipeline defined in `src/pipeline.py`
2. start a run and watch the steps
3. check the metrics
4. find the new version sitting in the registry, pending approval


## 1. Setup

The notebook runs from `notebooks/`, but the pipeline's
`source_dir="src"` and `code="src/..."` paths resolve against the repo
root.

In [1]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(os.getcwd())

/home/sagemaker-user/mlops-sagemaker-studio


The three values below come from Terraform:

```
terraform -chdir=infra output -raw data_bucket
terraform -chdir=infra output -raw alice_role_arn
terraform -chdir=infra output -raw model_package_group
```

In [2]:
import boto3

REGION = "ca-central-1"

BUCKET = "mlops-sagemaker-studio-dev-data-3vi8kw"
ROLE = "arn:aws:iam::099139718958:role/mlops-sagemaker-studio-dev-role-alice"
MODEL_PACKAGE_GROUP = "mlops-sagemaker-studio-dev-bike-sharing-rf"

sm = boto3.client("sagemaker", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)

sm.describe_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP)
print(f"registry group {MODEL_PACKAGE_GROUP} reachable")

registry group mlops-sagemaker-studio-dev-bike-sharing-rf reachable


## 2. Build and upsert

`build()` returns the Pipeline object; nothing exists in AWS until
`upsert()`. Upsert rather than create -- rerunning this cell is the
normal way to iterate on a definition.

In [3]:
import sys

# src/ is a directory of entry-point scripts, not a package -- adding an
# __init__.py would ship it to the container too.
sys.path.insert(0, "src")

from pipeline import PIPELINE_NAME, SKLEARN_IMAGE, build

pipeline = build(
    bucket=BUCKET,
    role=ROLE,
    model_package_group=MODEL_PACKAGE_GROUP,
    image=SKLEARN_IMAGE,
    instance_type="ml.m5.large",
    rmse_threshold=150.0,
)

pipeline.upsert(role_arn=ROLE)
print(f"upserted {PIPELINE_NAME}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


[08/01/26 21:37:10] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=122455;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=122456;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


                    INFO     StoppingCondition not provided. Using default:                         ]8;id=122463;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=122464;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

...

[08/01/26 21:37:12] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=122524;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=122525;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=122530;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=122531;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'CertifyForMarketplace' from the pipeline definition     ]8;id=122538;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py\model_step.py]8;;\:]8;id=122539;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py#195\195]8;;\
                             since it will be overridden in pipeline execution time.                               

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=122544;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=122545;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

upserted bike-sharing-rf


The definition is JSON underneath -- what the `aws_sagemaker_pipeline`
resource would have needed hand-written, and what the SDK generated
instead.

In [4]:
import json

raw = pipeline.definition()
definition = json.loads(raw)

print(f"{len(definition['Steps'])} top-level steps")
for step in definition["Steps"]:
    print(f"  {step['Name']:12} {step['Type']}")

print()
print(f"definition is {len(raw):,} characters of JSON")

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=122550;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=122551;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[08/01/26 21:37:13] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=122556;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=122557;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=122562;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=122563;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=122568;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=122569;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

4 top-level steps
  Preprocess   Processing
  Train        Training
  Evaluate     Processing
  CheckRmse    Condition

definition is 6,374 characters of JSON


## 3. Run it

This is the billed part: three jobs on `ml.m5.large`, a few minutes
each. Instances are provisioned and torn down per step -- nothing keeps
running afterwards, unlike the JupyterLab app.

In [5]:
execution = pipeline.start()

print(execution.arn)

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=122574;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=122575;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

arn:aws:sagemaker:ca-central-1:099139718958:pipeline/bike-sharing-rf/execution/gbc6zea5zf23


In [6]:
# Roughly 8-12 minutes cold; a rerun that hits the step cache is faster.
execution.wait()

print(execution.describe()["PipelineExecutionStatus"])

Succeeded


In [7]:
for step in execution.list_steps():
    status = step["StepStatus"]
    cached = " (cached)" if step.get("CacheHitResult", {}).get("SourcePipelineExecutionArn") else ""
    print(f"{step['StepName']:12} {status}{cached}")

Register     Succeeded
CheckRmse    Succeeded
Evaluate     Succeeded
Train        Succeeded
Preprocess   Succeeded


## 4. Metrics

The evaluate step wrote its report to S3. This is the same document the
gate reads.

In [8]:
obj = s3.get_object(Bucket=BUCKET, Key="model/evaluation/evaluation.json")
report = json.loads(obj["Body"].read())

metrics = report["regression_metrics"]
rmse = metrics["rmse"]["value"]
baseline = metrics["baseline_rmse"]["value"]

print(f"rmse={rmse:.4f} r2={metrics['r2']['value']:.4f}")
print(f"baseline rmse={baseline:.4f}")

assert rmse < baseline, "model is no better than predicting the mean"

rmse=126.3496 r2=0.6342
baseline rmse=227.8080


## 5. The gate

The run above passed, so a version was registered -- pending, not
approved.

In [9]:
def list_versions():
    """Every version, newest first. Paginated: the API caps a page at 100."""
    pages = sm.get_paginator("list_model_packages").paginate(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
        SortBy="CreationTime",
        SortOrder="Descending",
    )
    return [v for page in pages for v in page["ModelPackageSummaryList"]]


versions = list_versions()

for v in versions:
    print(f"v{v['ModelPackageVersion']:<3} {v['ModelApprovalStatus']:<22} {v['CreationTime']:%Y-%m-%d %H:%M}")

latest = versions[0]
assert latest["ModelApprovalStatus"] == "PendingManualApproval", latest["ModelApprovalStatus"]
print()
print("newest version is pending -- nothing deploys it until approved")

v1   PendingManualApproval  2026-08-01 21:44

newest version is pending -- nothing deploys it until approved


In [10]:
strict = pipeline.start(parameters={"RmseThreshold": 50.0})
strict.wait()

print(strict.describe()["PipelineExecutionStatus"])

steps = {s["StepName"]: s["StepStatus"] for s in strict.list_steps()}
print(steps)

assert "Register" not in steps, "register ran despite failing the gate"

after = list_versions()
assert len(after) == len(versions), "a version was registered despite failing the gate"

print()
print(f"run succeeded, register skipped, still {len(after)} version(s)")

[08/01/26 21:45:15] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=122580;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=122581;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

Succeeded
{'CheckRmse': 'Succeeded', 'Evaluate': 'Succeeded', 'Train': 'Succeeded', 'Preprocess': 'Succeeded'}

run succeeded, register skipped, still 1 version(s)


## 6. Approve

Approval is deliberately a separate act from training.

Studio: **Models > Model registry >** the group **>** the version **>
Update status**. Or from here:

In [11]:
sm.update_model_package(
    ModelPackageArn=latest["ModelPackageArn"],
    ModelApprovalStatus="Approved",
)

print(f"v{latest['ModelPackageVersion']} approved")

v1 approved
